# Reading a floor plan: depth, escape distance, critical doors

Companion to [Week 06 §11](../README.md#11-graphs-in-architecture-three-applications)
and `Lecture_06/graph_applications/plan_analysis.py`. The previous notebook
ran breadth-first search on a seven-room apartment. Here the same search,
on a small gallery building, answers three questions a reviewer asks of a
plan: **where do people pass, can everyone get out, and which door must
never be blocked?**

In [ ]:
# Setup: run this cell first. It finds the shared drawing helper (viz.py),
# whether this notebook sits in Lecture/notebooks/ or is your copy in MyWork/.
import sys
import pathlib

HERE = pathlib.Path.cwd()
NB = next(p / "Lecture" / "notebooks" for p in [HERE, *HERE.parents]
          if (p / "Lecture" / "notebooks" / "viz.py").exists())
sys.path.insert(0, str(NB))
from viz import chart, draw_graph, draw_mesh, draw_plan, draw_points, ramp, row, view3d

import heapq
import math
from collections import deque

from compas.datastructures import Graph

## Step 1 — the plan is rectangles
Each room is `(xmin, ymin, xmax, ymax)` in metres. No graph yet: this is
what a drawing gives you.

In [ ]:
ROOMS = {
    "lobby": (0, 0, 6, 6), "cafe": (6, 0, 12, 6), "shop": (12, 0, 16, 6), "office": (16, 0, 20, 6),
    "wc": (20, 0, 24, 3), "stair": (20, 3, 24, 6),
    "corr_w": (0, 6, 8, 8), "corr_m": (8, 6, 16, 8), "corr_e": (16, 6, 24, 8),
    "gallery1": (0, 8, 6, 16), "gallery2": (6, 8, 12, 16), "gallery3": (12, 8, 18, 16),
    "gallery4": (18, 8, 24, 16),
}
EXITS = ["lobby", "stair"]
draw_plan(ROOMS, "a gallery building, 24 m by 16 m")

## Step 2 — which rooms share a wall?
Two rectangles share a wall when they touch along a segment longer than a
door. That is two `max`/`min` pairs. The result is every place a door
**could** be.

In [ ]:
def shared_wall(a, b, rooms=ROOMS):
    """The two ends of the wall rooms a and b share, or None."""
    ax0, ay0, ax1, ay1 = rooms[a]
    bx0, by0, bx1, by1 = rooms[b]
    x0, x1 = max(ax0, bx0), min(ax1, bx1)
    y0, y1 = max(ay0, by0), min(ay1, by1)
    if x0 > x1 or y0 > y1 or (x1 - x0) + (y1 - y0) < 1.0:
        return None
    return (x0, y0), (x1, y1)


names = list(ROOMS)
walls = [(a, b) for i, a in enumerate(names) for b in names[i + 1:] if shared_wall(a, b)]
print(len(ROOMS), "rooms,", len(walls), "shared walls")
print("the lobby touches:", [b if a == "lobby" else a for a, b in walls if "lobby" in (a, b)])

## Step 3 — doors make it a graph
The architect put a door in 16 of those walls. A node is a room (at its
centre); an edge is a door, and it stores two things: where the door is,
and the **walking length** centre → door → centre.

In [ ]:
DOORS = [
    ("lobby", "cafe"), ("lobby", "corr_w"), ("cafe", "corr_m"), ("shop", "corr_m"),
    ("shop", "office"), ("office", "wc"), ("stair", "corr_e"), ("corr_w", "corr_m"),
    ("corr_m", "corr_e"), ("gallery1", "corr_w"), ("gallery2", "corr_m"), ("gallery3", "corr_m"),
    ("gallery4", "corr_e"), ("gallery1", "gallery2"), ("gallery2", "gallery3"), ("gallery3", "gallery4"),
]


def plan_graph(doors):
    graph = Graph()
    for name, (x0, y0, x1, y1) in ROOMS.items():
        graph.add_node(name, x=(x0 + x1) / 2, y=(y0 + y1) / 2, z=0.0, area=(x1 - x0) * (y1 - y0))
    for a, b in doors:
        (x0, y0), (x1, y1) = shared_wall(a, b)
        door = [(x0 + x1) / 2, (y0 + y1) / 2, 0.0]
        length = math.dist(graph.node_coordinates(a), door) + math.dist(door, graph.node_coordinates(b))
        graph.add_edge(a, b, door=door, length=length)
    return graph


plan = plan_graph(DOORS)
print(plan.number_of_nodes(), "rooms,", plan.number_of_edges(), "doors, out of", len(walls), "possible")
print("lobby -> cafe is", round(plan.edge_attribute(("lobby", "cafe"), "length"), 1), "m on foot")
draw_plan(ROOMS, "rooms and doors", graph=plan)

## Step 4 — depth from the entrance
Breadth-first search from the lobby, exactly as in the previous notebook.
Redrawn with every room placed at its depth, the plan becomes what space
syntax calls a **justified graph**: the building as a visitor meets it.

In [ ]:
def hops_from(graph, start):
    hops = {start: 0}
    queue = deque([start])
    while queue:
        room = queue.popleft()
        for other in graph.neighbors(room):
            if other not in hops:
                hops[other] = hops[room] + 1
                queue.append(other)
    return hops


depth = hops_from(plan, "lobby")
for d in range(max(depth.values()) + 1):
    print(f"{d} doors from the lobby:", [room for room in depth if depth[room] == d])

justified = plan.copy()
for d in range(max(depth.values()) + 1):
    level = [room for room in depth if depth[room] == d]
    for i, room in enumerate(level):
        justified.node_attributes(room, "xy", [3.0 * (i - (len(level) - 1) / 2), 2.0 * d])
row(draw_plan(ROOMS, "depth from the lobby", size=(520, 380),
              fills={room: ramp(d / max(depth.values())) for room, d in depth.items()},
              notes={room: str(d) for room, d in depth.items()}),
    draw_graph(justified, "the same, justified", size=(460, 380),
               node_colors={room: ramp(d / max(depth.values())) for room, d in depth.items()}))

## Step 5 — integration: depth from *everywhere*
Depth from the lobby is one room's point of view. Run the search from every
room and average: a room with a low **mean depth** is close to everything
(people pass through it), a room with a high one is private, or forgotten.
Space syntax (Hillier & Hanson, 1984) calls the inverse *integration*.

In [ ]:
k = plan.number_of_nodes()
mean_depth = {room: sum(hops_from(plan, room).values()) / (k - 1) for room in plan.nodes()}
for room in sorted(mean_depth, key=mean_depth.get):
    print(f"{room:9s} mean depth {mean_depth[room]:.2f}  " + "#" * round(10 * mean_depth[room]))

lo, hi = min(mean_depth.values()), max(mean_depth.values())
draw_plan(ROOMS, "mean depth: blue = integrated, orange = segregated",
          fills={room: ramp((md - lo) / (hi - lo)) for room, md in mean_depth.items()},
          notes={room: f"{md:.2f}" for room, md in mean_depth.items()})

The middle corridor is the most integrated room, which is what a corridor
is for. Look at the **lobby**: the front door of the building is in the
deeper half of the list. Is that intended, or a problem? The number does
not answer that. It makes you ask.

## Step 6 — escape distance: now the metres matter
Counting doors is not enough for a fire code, which limits the **walking
distance** to an exit. Breadth-first search cannot do this: it treats a 2 m
step and a 12 m step alike. Dijkstra's algorithm is the same search with a
priority queue: always continue from the room that is *nearest so far*.
Starting it from all exits at once gives the distance to the nearest one.

In [ ]:
def escape_distance(graph, exits):
    dist = {e: 0.0 for e in exits}
    queue = [(0.0, e) for e in exits]
    done = set()
    while queue:
        d, room = heapq.heappop(queue)              # the nearest room not yet settled
        if room in done:
            continue
        done.add(room)
        for other in graph.neighbors(room):
            edge = (room, other) if graph.has_edge((room, other)) else (other, room)
            reached = d + graph.edge_attribute(edge, "length")
            if reached < dist.get(other, float("inf")):
                dist[other] = reached
                heapq.heappush(queue, (reached, other))
    return dist


LIMIT = 25.0
escape = escape_distance(plan, EXITS)
for room, d in sorted(escape.items(), key=lambda item: -item[1])[:5]:
    print(f"{room:9s} {d:5.1f} m" + ("   <-- over the limit" if d > LIMIT else ""))
draw_plan(ROOMS, f"metres to the nearest exit (limit {LIMIT:.0f} m)",
          fills={room: "#ffc9c9" if d > LIMIT else ramp(0.5 * d / LIMIT) for room, d in escape.items()},
          notes={room: f"{d:.1f} m" for room, d in escape.items()})

The WC is right next to the fire stair on the drawing, and the farthest
room from an exit in the graph, because its only door opens the other way.
**Near in space is not near in the graph.**

## Step 7 — which doors must never be blocked?
Remove one door, count the pieces, put it back. A door whose removal splits
the plan is a single point of failure (a *bridge*, in graph terms).

In [ ]:
def pieces(graph):
    seen, count = set(), 0
    for room in graph.nodes():
        if room not in seen:
            seen |= set(hops_from(graph, room))
            count += 1
    return count


def critical_doors(graph):
    critical = []
    for edge in list(graph.edges()):
        trial = graph.copy()
        trial.delete_edge(edge)
        if pieces(trial) > 1:
            critical.append(edge)
    return critical


critical = critical_doors(plan)
print(len(critical), "critical doors:", critical)
draw_plan(ROOMS, "critical doors in orange", graph=plan, edge_colors={edge: "#e8590c" for edge in critical})

One of them is the only door to the fire stair.

## Step 8 — a design move, measured
The analysis is a function of the door list, so a change to the plan is a
change to one list. Add a single door between the WC and the stair, and
compare.

In [ ]:
better = plan_graph(DOORS + [("wc", "stair")])
before, after = escape, escape_distance(better, EXITS)
print(f"{'':22s} {'before':>8s} {'after':>8s}")
print(f"{'farthest room':22s} {max(before.values()):7.1f}m {max(after.values()):7.1f}m")
print(f"{'rooms over the limit':22s} {sum(d > LIMIT for d in before.values()):8d} {sum(d > LIMIT for d in after.values()):8d}")
print(f"{'critical doors':22s} {len(critical):8d} {len(critical_doors(better)):8d}")
draw_plan(ROOMS, "with a door from the WC to the stair",
          fills={room: ramp(0.5 * d / LIMIT) for room, d in after.items()},
          notes={room: f"{d:.1f} m" for room, d in after.items()},
          graph=better, edge_colors={edge: "#e8590c" for edge in critical_doors(better)})

## Your turn
1. Which **single** extra door lowers the lobby's mean depth the most? Loop
   over the unused entries of `walls` and measure each.
2. Close the door between `corr_m` and `corr_e` (a fire door, shut). What
   happens to the escape distances, and to the number of pieces?
3. Make the stair no longer an exit (`EXITS = ["lobby"]`). Which rooms now
   break the limit?
4. `plan_analysis.py` normalises mean depth into *relative asymmetry*. Read
   its `integration()` and say when that extra step is needed.